# 13주 · 무차원군·수리학적 상사

수리학 CE272 · 주차별 공통 실습

핵심: 원리·식의 의미·손계산·조건 변경·결과 해석. 입력 단위와 지배 방정식·경계조건부터 확인

학습 주제: 반복변수법·무차원군 / 기하·운동·동역학적 상사

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/week13/CE272_W13.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE272-hydraulics/week13) · [계산 도구·가정](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/METHODS.md)

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)

실행: 처음에는 위에서 아래로 셀 실행. 공통 준비 셀 실행 후 각 S절 개별 실행 가능

진행: 예측 → 손계산·식 설정 → 실행 → 조건 하나 변경 → 원리·가정·결과 설명

Excel 중점: 차원 지수 검산·원본 상사 예제

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import importlib.util
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.integrate import trapezoid

# 로컬 공통 자료에서는 옆 과목 파일 사용. Colab에서는 같은 공개 모듈 불러오기
module_path = next((p for p in [Path.cwd()/"hydraulics.py", Path.cwd().parent/"hydraulics.py"] if p.exists()), None)
if module_path is None:
    module_path = Path(tempfile.gettempdir())/"lecture_ce272_hydraulics.py"
    urlretrieve("https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/hydraulics.py",module_path)
spec = importlib.util.spec_from_file_location("lecture_ce272_hydraulics",module_path)
hyd = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hyd)
print("수리학 공통 계산 도구 준비 완료. SI 단위·Darcy 계수 사용")

## S01. 전단응력·압력경사와 Blasius 관계

**슬라이드 연결**: CE272_W13a_Dimensional_Analysis_2.pptx p12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24 / CE272_W13b_Hydraulic_Similitude_1.pptx p5, 6, 7, 8

**자료 구분**: 원본 무차원화·공식 재현

**먼저 예측**: 무차원화가 가능하다는 사실만으로 Blasius의 −1/4 지수도 얻을 수 있는가?

**문제**: 원본의 Π₁=(Δp/L)D/(ρV²), Π₂=Re 관계에서 Π₁=0.1582 Re⁻⁰·²⁵와 Darcy f=0.3164 Re⁻⁰·²⁵의 연결을 확인하시오. 아래 점은 관측자료가 아니라 공식으로 생성한 재현점임을 구분하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교

In [ ]:
Re=np.logspace(np.log10(4001),5,30)
pi1=.1582*Re**(-.25)
f_blasius=.3164*Re**(-.25)
display(pd.DataFrame({'Re':Re,'pi1_formula_points':pi1,'Darcy_f':f_blasius,'f_over_2':f_blasius/2}).head())
plt.loglog(Re,pi1,label='Pi1');plt.loglog(Re,f_blasius,label='Darcy f');plt.xlabel('Re');plt.legend();plt.grid();plt.show()

**검토 문제**: 계수·지수가 실험과 유효 범위에 의존함을 설명하시오. 거친 관이나 층류에 같은 상관식을 적용할 수 없는 이유를 제시하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분

## S02. 기하학적 상사와 시간비

**슬라이드 연결**: CE272_W13b_Hydraulic_Similitude_1.pptx p12, 13, 15

**자료 구분**: 원본 세 상사 예제

**먼저 예측**: 축척 1:10이라는 정보만으로 유량비 1:1000을 확정할 수 있는가?

**문제**: Lr=10, Qm=0.1 m³/s인 원본 예제에서 Tr=1을 추가 가정한 Qp를 구하시오. 원본의 Lr=20, Qp=100, Qm=0.1 조건에서 Tr를 구하시오. Lr=10, Vr=2, Pp=80 kW, ρr=1일 때 모형 동력을 구하시오. 모든 비는 원형/모형이다.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교

In [ ]:
geometric_case=hyd.model_ratios(10,1)
Tr_second=20**3/(100/.1)
power_case=hyd.model_ratios(10,10/2)
pd.Series({'Qp_with_Tr_1_m3_s':.1*geometric_case['Qr'],
           'Tr_second_example':Tr_second,
           'Pm_third_example_kW':80/power_case['Pr_power'],
           'Qp_if_Froude_time_ratio_m3_s':.1*hyd.model_ratios(10,np.sqrt(10))['Qr']})

**검토 문제**: 기하학적 상사만으로 시간비가 정해지지 않음을 설명하시오. 원형/모형 비의 방향을 뒤집었을 때 생기는 계산 오류도 예로 드시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분

## 학습 기록

무차원화가 알려주는 관계와 추가 실험·가정이 필요한 부분을 구분하시오.

선택 심화: 개념 학습 후 AI의 설명·계산 검토. 단위·가정·보존식으로 확인한 근거와 수정한 부분 기록

In [ ]:
prediction = None
hand_calculation = None
changed_condition = None
principle_and_limits = None
records = {'예측':prediction,'손계산':hand_calculation,'조건 변경':changed_condition,'원리·한계':principle_and_limits}
missing = [k for k,v in records.items() if v is None or str(v).strip() == '']
print('작성 필요:',missing) if missing else print('기록 항목 작성 완료. 계산·논리는 별도 검토 필요')